<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  13. Prompt Templates
</h2>

In [3]:
%pip install --upgrade mistralai


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os

load_dotenv()
mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

### **13.1 Create Prompt**

In [5]:
prompt_text = """
Context: You are a senior Python developer
Task: Your task is to review the code from the user and make it compliant with PEP8
Format & Style: Consise and straight to the point

Steps / Rules:
- Think step-by-step before answering.
- Highlight any potential edge cases or missing details.
- Keep explanations clear and actionable.

Code:
{{code}}
"""

prompt_object = mistral.beta.prompts.create(
    name="pep8_review", # stable object name (must be unique)
    title="PEP 8 Review", # Display title
    description="A prompt to get Python code reviewed and updated to. PEP8 standards", # Display description
    definition={"content": prompt_text, "variables": [{"name": "code"}]},
    aliases=["dev"] # tags
)

print(prompt_object.model_dump_json(indent=2))


{
  "id": "019f477b-ada2-7214-b78f-15bed191c6bb",
  "name": "pep8_review",
  "definition": {
    "content": "\nContext: You are a senior Python developer\nTask: Your task is to review the code from the user and make it compliant with PEP8\nFormat & Style: Consise and straight to the point\n\nSteps / Rules:\n- Think step-by-step before answering.\n- Highlight any potential edge cases or missing details.\n- Keep explanations clear and actionable.\n\nCode:\n{{code}}\n",
    "variables": [
      {
        "name": "code"
      }
    ]
  },
  "version": 1,
  "notes": "",
  "aliases": [
    "dev"
  ],
  "sharingScope": "private",
  "createdAt": "2026-07-09T15:25:15.554774Z",
  "updatedAt": "2026-07-09T15:25:15.554774Z",
  "latestVersion": 1,
  "title": "PEP 8 Review",
  "description": "A prompt to get Python code reviewed and updated to. PEP8 standards"
}


### **13.2 Get & Use Prompt**

In [ ]:
# Can also specify an alias to grab the correct version
prompt = mistral.beta.prompts.get(
    prompt_id=prompt_object.id
)

In [7]:
formatted_prompt = prompt.definition.content.replace(
    "{{code}}",
    """
    def MyFunction(x):
        return x
    """
)

formatted_prompt

'\nContext: You are a senior Python developer\nTask: Your task is to review the code from the user and make it compliant with PEP8\nFormat & Style: Consise and straight to the point\n\nSteps / Rules:\n- Think step-by-step before answering.\n- Highlight any potential edge cases or missing details.\n- Keep explanations clear and actionable.\n\nCode:\n\n    def MyFunction(x):\n        return x\n    \n'

In [8]:
from IPython.display import Markdown
from mistralai.client.models import UserMessage

response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=[UserMessage(content=formatted_prompt)]
)

Markdown(response.choices[0].message.content)

Here's the PEP8-compliant version of your code with explanations:

```python
def my_function(x):
    return x
```

Key changes made:
1. Renamed `MyFunction` to `my_function` (PEP8: function names should be lowercase with words separated by underscores)
2. Added consistent spacing around the `=` in the function definition (PEP8: spaces around operators)

Edge cases/considerations:
- If this is part of a public API, you might want to keep the original name (but this would violate PEP8)
- If `x` is meant to be a specific type, consider adding type hints (PEP484)
- If this is a method in a class, the naming convention would be different (snake_case for methods)

The changes are minimal since the original code was already quite clean. The main issue was the naming convention.

### **13.3 List Prompts**

In [9]:
response = mistral.beta.prompts.list()
response

PromptsListResponse(next=<function BetaPrompts.list.<locals>.next_func at 0x112defc10>, result=ListPromptsResponse(data=[Prompt(id='019f477b-ada2-7214-b78f-15bed191c6bb', name='pep8_review', definition=None, version=1, notes='', aliases=['dev'], sharing_scope='private', created_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), latest_version=1, title='PEP 8 Review', description='A prompt to get Python code reviewed and updated to. PEP8 standards')], next_page_token=''))

In [10]:
response.result.data

[Prompt(id='019f477b-ada2-7214-b78f-15bed191c6bb', name='pep8_review', definition=None, version=1, notes='', aliases=['dev'], sharing_scope='private', created_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), latest_version=1, title='PEP 8 Review', description='A prompt to get Python code reviewed and updated to. PEP8 standards')]

In [11]:
prompts = [prompt for prompt in response.result.data]
prompts

[Prompt(id='019f477b-ada2-7214-b78f-15bed191c6bb', name='pep8_review', definition=None, version=1, notes='', aliases=['dev'], sharing_scope='private', created_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), latest_version=1, title='PEP 8 Review', description='A prompt to get Python code reviewed and updated to. PEP8 standards')]

In [12]:
for prompt in prompts:
    print(prompt)

id='019f477b-ada2-7214-b78f-15bed191c6bb' name='pep8_review' definition=None version=1 notes='' aliases=['dev'] sharing_scope='private' created_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)) updated_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)) latest_version=1 title='PEP 8 Review' description='A prompt to get Python code reviewed and updated to. PEP8 standards'


### **13.4 Update Prompt**

In [13]:
# Does not create a new version
mistral.beta.prompts.update_metadata(
    prompt_id=prompt.id,
    title="new title",
    description="new description"
)

Prompt(id='019f477b-ada2-7214-b78f-15bed191c6bb', name='pep8_review', definition=PromptDefinition(content='\nContext: You are a senior Python developer\nTask: Your task is to review the code from the user and make it compliant with PEP8\nFormat & Style: Consise and straight to the point\n\nSteps / Rules:\n- Think step-by-step before answering.\n- Highlight any potential edge cases or missing details.\n- Keep explanations clear and actionable.\n\nCode:\n{{code}}\n', variables=[PromptVariable(name='code')]), version=1, notes='', aliases=['dev'], sharing_scope='private', created_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 7, 9, 15, 28, 50, 278536, tzinfo=TzInfo(0)), latest_version=1, title='new title', description='new description')

In [ ]:
# Creates a new version
mistral.beta.prompts.create_version(
    prompt_id=prompt.id,
    definition={"content": "NEW VERSION", "variables": []},
    aliases=["prod"]
)

CreatePromptVersionResponse(version=2, deduplicated=False)

### **13.5 Get Specific Version**

In [15]:
versions = mistral.beta.prompts.list_versions(
    prompt_id=prompt.id
)

print(versions.model_dump_json(indent=2))

{
  "data": [
    {
      "version": 2,
      "definition": {
        "content": "NEW VERSION",
        "variables": []
      },
      "notes": "",
      "aliases": [
        "prod"
      ],
      "createdAt": "2026-07-09T15:29:36.944549Z"
    },
    {
      "version": 1,
      "definition": {
        "content": "\nContext: You are a senior Python developer\nTask: Your task is to review the code from the user and make it compliant with PEP8\nFormat & Style: Consise and straight to the point\n\nSteps / Rules:\n- Think step-by-step before answering.\n- Highlight any potential edge cases or missing details.\n- Keep explanations clear and actionable.\n\nCode:\n{{code}}\n",
        "variables": [
          {
            "name": "code"
          }
        ]
      },
      "notes": "",
      "aliases": [
        "dev"
      ],
      "createdAt": "2026-07-09T15:25:15.554774Z"
    }
  ]
}


In [ ]:
mistral.beta.prompts.get_version(
    prompt_id=prompt.id,
    version=1
)

Prompt(id='019f477b-ada2-7214-b78f-15bed191c6bb', name='pep8_review', definition=PromptDefinition(content='\nContext: You are a senior Python developer\nTask: Your task is to review the code from the user and make it compliant with PEP8\nFormat & Style: Consise and straight to the point\n\nSteps / Rules:\n- Think step-by-step before answering.\n- Highlight any potential edge cases or missing details.\n- Keep explanations clear and actionable.\n\nCode:\n{{code}}\n', variables=[PromptVariable(name='code')]), version=1, notes='', aliases=['dev'], sharing_scope='private', created_at=datetime.datetime(2026, 7, 9, 15, 25, 15, 554774, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 7, 9, 15, 29, 36, 957064, tzinfo=TzInfo(0)), latest_version=2, title='new title', description='new description')